# Inspire-HEP API — Searching, Fetching, and Verifying Papers

> **Bias note:** Inspire-HEP is the standard literature database in high-energy physics.
> Everything in this notebook reflects that community's tools and conventions.
> If your field uses a different database (ADS for astrophysics, SPIRES for older HEP, etc.),
> the concepts transfer but the endpoints will differ.

This notebook demonstrates four things using the
[Inspire-HEP REST API](https://inspirehep.net/api/) — no authentication required:

1. **Searching** by keyword, author, or Inspire query syntax.
2. **Fetching** a specific paper's metadata by its ArXiv identifier.
3. **Retrieving BibTeX** — the standard citation format in HEP.
4. **Verifying that a paper exists** — a practical sanity check for LLM-generated citations.

## Setup

`requests` is included in Nestling's dependencies and needs no extra installation.

In [ ]:
import requests

INSPIRE_API = "https://inspirehep.net/api"

## 1. Keyword Search

Inspire-HEP supports a rich query language.
The most useful field qualifiers for everyday use are:

| Qualifier | Matches |
| --- | --- |
| `a <name>` | Author name |
| `t <word>` | Title word |
| `k <word>` | Keyword |
| `eprint <id>` | ArXiv identifier |
| `texkey <key>` | Inspire texkey (e.g. `ATLAS:2012yve`) |
| `j <journal>` | Journal name |

Full query syntax: https://inspirehep.net/literature?ui-citation-summary=true

In [ ]:
def search_inspire(query: str, max_results: int = 5) -> list[dict]:
    """Search Inspire-HEP and return a list of metadata records."""
    response = requests.get(
        f"{INSPIRE_API}/literature",
        params={"q": query, "size": max_results, "sort": "mostrecent"},
        timeout=10,
    )
    response.raise_for_status()
    return response.json().get("hits", {}).get("hits", [])


def print_inspire_record(meta: dict) -> None:
    """Pretty-print one Inspire metadata record."""
    title = meta.get("titles", [{}])[0].get("title", "N/A")
    texkey = meta.get("texkeys", ["N/A"])[0]
    citations = meta.get("citation_count", 0)
    arxiv_id = meta.get("arxiv_eprints", [{}])[0].get("value", "N/A")
    journal = ""
    if meta.get("publication_info"):
        pub = meta["publication_info"][0]
        journal = f"{pub.get('journal_title', '')} {pub.get('journal_volume', '')} ({pub.get('year', '')})"
    print(f"Title    : {title}")
    print(f"Texkey   : {texkey}")
    print(f"ArXiv    : {arxiv_id}")
    print(f"Citations: {citations}")
    if journal.strip():
        print(f"Journal  : {journal.strip()}")
    print()


# Search for papers by the author — a natural starting point for a new group member
records = search_inspire("a Meighen-Berger", max_results=5)
print(f"Found {len(records)} result(s)\n")
for hit in records:
    print_inspire_record(hit["metadata"])

## 2. Fetching a Specific Paper by ArXiv ID

If you know the ArXiv identifier, you can retrieve the full Inspire record directly.
This is useful when you want citation counts, the published journal reference,
or the texkey — all of which ArXiv itself does not provide.

We use three recent papers as benchmarks:

| ArXiv ID | Description |
| --- | --- |
| `2605.16721` | Beacom et al. — CP-violating phase with atmospheric neutrinos |
| `2512.18093` | Meighen-Berger et al. — dark matter & ultrahigh-energy cosmic rays (PRD) |
| `2511.13111` | Orsoe et al. — NuBench benchmark for neutrino telescopes (JINST) |

Note: very recent preprints (days to weeks old) may not yet appear in Inspire,
which indexes with a short delay. A miss on a brand-new paper is not necessarily a problem.

In [ ]:
def fetch_by_arxiv_id(arxiv_id: str) -> dict | None:
    """Fetch the Inspire metadata record for a paper by its ArXiv identifier."""
    response = requests.get(
        f"{INSPIRE_API}/literature",
        params={"q": f"eprint {arxiv_id}", "size": 1},
        timeout=10,
    )
    response.raise_for_status()
    hits = response.json().get("hits", {}).get("hits", [])
    return hits[0] if hits else None


BENCHMARK_PAPERS = [
    ("2605.16721", "Beacom et al. — atmospheric neutrinos"),
    ("2512.18093", "Meighen-Berger et al. — dark matter & UHECR"),
    ("2511.13111", "Orsoe et al. — NuBench"),
]

for arxiv_id, label in BENCHMARK_PAPERS:
    hit = fetch_by_arxiv_id(arxiv_id)
    if hit:
        print(f"[{label}]")
        print_inspire_record(hit["metadata"])
    else:
        print(f"[{label}]")
        print(f"  arXiv:{arxiv_id} not yet indexed in Inspire (may be too recent)\n")

## 3. Retrieving BibTeX Citations

In HEP, Inspire-HEP is the standard source for BibTeX entries.
The texkey — a short identifier like `Meighen-Berger:2025hrq` or `ATLAS:2012yve` — is what you
put in your `\cite{}` commands in LaTeX.

Inspire generates canonical BibTeX entries that include the journal reference, DOI,
and arXiv eprint number. Always use Inspire BibTeX rather than typing entries by hand —
it avoids transcription errors and keeps your citations consistent with the rest of the field.

The API returns BibTeX directly when you request it with the right `Accept` header.

In [ ]:
def get_bibtex(arxiv_id: str) -> str | None:
    """Retrieve the Inspire BibTeX entry for a paper by its ArXiv identifier."""
    # Step 1: find the numeric Inspire literature ID
    response = requests.get(
        f"{INSPIRE_API}/literature",
        params={"q": f"eprint {arxiv_id}", "size": 1, "fields": "id"},
        timeout=10,
    )
    response.raise_for_status()
    hits = response.json().get("hits", {}).get("hits", [])
    if not hits:
        return None
    inspire_id = hits[0]["id"]

    # Step 2: fetch the BibTeX representation
    bibtex_response = requests.get(
        f"{INSPIRE_API}/literature/{inspire_id}",
        headers={"Accept": "application/x-bibtex"},
        timeout=10,
    )
    bibtex_response.raise_for_status()
    return bibtex_response.text


# Fetch BibTeX for the two published benchmark papers
for arxiv_id in ["2512.18093", "2511.13111"]:
    bibtex = get_bibtex(arxiv_id)
    if bibtex:
        print(bibtex)
    else:
        print(f"arXiv:{arxiv_id} not yet in Inspire\n")

## 4. Hallucination Check for LLM-Generated Citations

Large language models are useful tools for literature exploration — but they sometimes
**hallucinate citations**: they produce paper titles, author lists, and ArXiv IDs that look
completely plausible but do not actually exist.

A fast sanity check: query Inspire (or ArXiv) with the ArXiv ID or texkey the LLM gave you.

- **Hit with matching title** → strong evidence the paper is real.
- **No hit** → proceed with caution. The paper may be:
  - Hallucinated (most common).
  - Too recent to be indexed yet (Inspire lags by days to weeks).
  - In a field Inspire does not cover (non-HEP).

Always verify before citing — a hallucinated reference in a submitted paper is embarrassing
and, if missed by reviewers, can cause lasting confusion in the literature.

In [ ]:
def verify_paper(arxiv_id: str) -> tuple[bool, str]:
    """Check whether a paper with the given ArXiv ID exists in Inspire-HEP.

    Returns (found: bool, title: str) — title is empty string if not found.
    """
    response = requests.get(
        f"{INSPIRE_API}/literature",
        params={"q": f"eprint {arxiv_id}", "size": 1},
        timeout=10,
    )
    response.raise_for_status()
    hits = response.json().get("hits", {}).get("hits", [])
    if hits:
        title = hits[0]["metadata"].get("titles", [{}])[0].get("title", "")
        return True, title
    return False, ""


print("=== Real papers (should be found) ===\n")
# The two published papers are definitely in Inspire
for arxiv_id, label in [
    ("2512.18093", "Meighen-Berger et al. PRD"),
    ("2511.13111", "Orsoe et al. JINST"),
]:
    found, title = verify_paper(arxiv_id)
    status = f"FOUND ✓  →  {title[:80]}" if found else "NOT FOUND"
    print(f"  arXiv:{arxiv_id}  [{label}]")
    print(f"  {status}\n")

# The very recent preprint may or may not be indexed yet
for arxiv_id, label in [("2605.16721", "Beacom et al. — very recent preprint")]:
    found, title = verify_paper(arxiv_id)
    if found:
        status = f"FOUND ✓  →  {title[:80]}"
    else:
        status = "NOT FOUND (paper is very recent — Inspire may not have indexed it yet)"
    print(f"  arXiv:{arxiv_id}  [{label}]")
    print(f"  {status}\n")

print("=== Hallucinated paper (should not be found) ===\n")
fake_id = "2501.99999"
found, title = verify_paper(fake_id)
status = f"FOUND ✓  →  {title}" if found else "NOT FOUND ✗  →  likely hallucinated"
print(f"  arXiv:{fake_id}  [LLM-generated, does not exist]")
print(f"  {status}\n")

## 5. Try It Yourself

Paste an ArXiv ID or texkey from a paper an LLM gave you, or from a reference list you are checking.

In [ ]:
# Paste a paper ID to verify, or a search query to explore
CHECK_ID = "YOUR_ARXIV_ID_HERE"  # e.g. "2312.01234"
MY_QUERY = "a YOUR_SUPERVISOR_NAME"  # e.g. "a Particle AND k neutrino"

if CHECK_ID != "YOUR_ARXIV_ID_HERE":
    found, title = verify_paper(CHECK_ID)
    print(f"arXiv:{CHECK_ID}")
    print(f"  {'FOUND: ' + title if found else 'NOT FOUND'}")

if MY_QUERY != "a YOUR_SUPERVISOR_NAME":
    print(f"\nSearch: {MY_QUERY}\n")
    hits = search_inspire(MY_QUERY, max_results=5)
    for hit in hits:
        print_inspire_record(hit["metadata"])